# Aprendizado de Máquina — Lista prática 05

## Métodos Não Paramétricos: Aspectos Teóricos

**Gabriel Sanfins** &nbsp;·&nbsp; gabrielsanfins@id.uff.br

---

Esta é a única lista do curso sem conjunto de dados: tudo aqui é simulação, porque
o assunto da aula é uma afirmação sobre **geometria em dimensão alta**, e geometria
se mede sorteando pontos.

> **a maldição da dimensionalidade não é uma metáfora. Ela é uma conta sobre
> volume, e você vai reproduzi-la em quatro medições.**

As duas primeiras batem com a fórmula até a terceira casa. A terceira não bate —
e a parte interessante é entender por quê.

Cada lacuna está marcada com `...`. Substitua **cada uma** pela sua resposta e
rode a célula.

---
## 1. Importando os pacotes

In [ ]:
import numpy as np
from matplotlib.pyplot import subplots
from scipy.special import gamma

from sklearn.neighbors import NearestNeighbors, KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import SplineTransformer

import warnings
warnings.filterwarnings("ignore")

---
## Exercício 1 — quase todo ponto está na casca

Comece pelo fato mais simples e mais brutal. Sorteie pontos no cubo $[0,1]^p$ e
conte quantos estão a menos de $\varepsilon$ de **alguma** face.

Um ponto está perto da borda se **pelo menos uma** das suas coordenadas for
menor que $\varepsilon$ ou maior que $1-\varepsilon$. A probabilidade de isso
**não** acontecer é $(1-2\varepsilon)^p$, uma coordenada de cada vez.

In [ ]:
rng = np.random.default_rng(2026)
eps = 0.05
dims = [1, 2, 5, 10, 20, 50]

print("   d    medido   previsto")
for d in dims:
    X = rng.uniform(0, 1, size=(20000, d))
    perto = np.mean((X < eps).any(axis=1) | (X > 1 - eps).any(axis=1))   # (a)
    previsto = 1 - (1 - 2 * eps) ** d                                    # (b)
    print(f"  {d:3d}   {perto:.4f}    {previsto:.4f}")

Deve imprimir:

```
   d    medido   previsto
    1   0.0983    0.1000
    2   0.1905    0.1900
    5   0.4149    0.4095
   10   0.6531    0.6513
   20   0.8780    0.8784
   50   0.9953    0.9948
```

A fórmula acerta em todas as dimensões, a menos do erro de Monte Carlo na
terceira casa.

A leitura: com uma casca de espessura $0{,}05$ — cinco por cento da largura do
cubo de cada lado — em $p=50$ **99,5% dos pontos estão nela**. O "interior" do
cubo, que em $p=1$ contém 90% da massa, praticamente deixa de existir.

Ou seja: em dimensão alta, praticamente toda observação é um ponto de
fronteira do domínio.

---
## Exercício 2 — o vizinho mais próximo fica longe

A nota estima o raio necessário para reunir $k$ vizinhos por
$\rho \approx (k/n)^{1/p}$, tratando o volume de uma bola de raio $\rho$ como
$\rho^p$. O volume verdadeiro tem uma constante:

$$V_p(\rho) = \frac{\pi^{p/2}}{\Gamma\!\left(\frac p2 + 1\right)}\,\rho^p .$$

Vamos medir a distância ao vizinho mais próximo e comparar com as **duas**
previsões.

In [ ]:
def volume_bola_unitaria(d):
    return np.pi ** (d / 2) / gamma(d / 2 + 1)                 # (a)


rng = np.random.default_rng(2026)
n = 1000

print("   d   medido   heuristica   com o volume da bola      V_d")
for d in dims:
    X = rng.uniform(0, 1, size=(n, d))
    # o vizinho mais proximo de cada ponto e o SEGUNDO da lista (o primeiro e ele mesmo)
    dist, _ = NearestNeighbors(n_neighbors=2).fit(X).kneighbors(X)
    medido = np.median(dist[:, 1])                             # (b)

    heuristica = (1 / n) ** (1 / d)
    Vd = volume_bola_unitaria(d)
    com_volume = (1 / (n * Vd)) ** (1 / d)                     # (c)

    print(f"  {d:3d}   {medido:.4f}    {heuristica:.4f}       {com_volume:.4f}"
          f"          {Vd:.3g}")

Deve imprimir:

```
   d   medido   heuristica   com o volume da bola      V_d
    1   0.0003    0.0010       0.0005          2
    2   0.0151    0.0316       0.0178          3.14
    5   0.1735    0.2512       0.1802          5.26
   10   0.5039    0.5012       0.4564          2.55
   20   1.0440    0.7079       0.8500          0.0258
   50   2.1206    0.8710       1.5676          1.73e-13
```

Três coisas aqui, e a terceira é a mais estranha.

**Primeiro**, a fórmula com o volume da bola é bem melhor que a heurística até
$p=5$: ali ela dá 0,180 contra os 0,174 medidos, um erro de 4%, enquanto a
heurística dá 0,251 e erra por 45%. Em $p=10$ as duas se cruzam e a heurística
acerta melhor (0,501 contra 0,504 medidos) — coincidência, e não uma virada de
tendência: é o ponto em que o fator $V_p$, que vinha corrigindo para baixo, passa
a ser compensado pelo efeito da borda descrito a seguir.

**Segundo**, de $p=10$ em diante as duas previsões ficam **abaixo** do medido, e a
distância chega a $2{,}12$ — maior que a diagonal de um cubo de lado 1 em $p=4$.
A causa é o Exercício 1: as duas fórmulas supõem que a bola de vizinhos cabe
dentro do cubo, e em $p=20$ nenhum ponto tem essa sorte, porque 88% deles estão
na casca. Sobra menos volume disponível do que a conta previa, e o raio real
precisa ser maior.

**Terceiro**, olhe a coluna $V_p$. O volume da bola unitária **cresce até $p=5$ e
depois desaba**: 5,26 em $p=5$, 2,55 em $p=10$, 0,026 em $p=20$, $10^{-13}$ em
$p=50$. Uma bola de raio 1 inscrita num cubo de lado 2 ocupa, em $p=50$, uma
fração de $10^{-28}$ do cubo. Praticamente todo o volume do cubo está nos cantos,
e é por isso que "esfera de vizinhos" deixa de ser uma boa imagem.

> **Sua vez.** Repita a medição com $n=10\,000$ em vez de $1\,000$. Em $p=1$ a
> distância cai por um fator de 10, como esperado. Em $p=50$, por quanto ela cai?

---
## Exercício 3 — a taxa $n^{-2/(2+p)}$, medida

Agora o teorema em si. Vamos medir o erro do KNN contra $n$, em três dimensões, e
estimar o expoente por regressão nos logaritmos: se $\mathrm{EQM}\approx C n^{a}$,
então $\log \mathrm{EQM} \approx \log C + a\log n$, e $a$ é a inclinação.

A função de regressão tem curvatura em **todas** as coordenadas e constante de
Lipschitz que não cresce com $p$:

$$r(x) = \frac{1}{\sqrt p}\sum_{j=1}^{p}\operatorname{sen}(2\pi x_j).$$

Como conhecemos $r$, medimos o erro contra ela e não contra $y$ — o $\sigma^2$ é
uma constante aditiva que esconderia o expoente.

In [ ]:
def r(X):
    return np.sin(2 * np.pi * X).sum(axis=1) / np.sqrt(X.shape[1])


SIGMA = 0.3
ns = np.array([250, 500, 1000, 2000, 4000])
ks = np.array([1, 2, 3, 5, 8, 13, 21, 34, 55, 89])

resultados = {}
for d in (1, 5, 10):
    rng = np.random.default_rng(2026)
    eqm = []
    for n in ns:
        acc = []
        for _ in range(4):
            X = rng.uniform(0, 1, size=(n, d))
            y = r(X) + rng.normal(0, SIGMA, size=n)
            X0 = rng.uniform(0, 1, size=(2000, d))
            r0 = r(X0)                                          # (a) o alvo verdadeiro
            # o melhor k possivel: medimos a taxa do metodo, sem o erro de escolher k
            acc.append(min(
                np.mean((KNeighborsRegressor(n_neighbors=int(k)).fit(X, y).predict(X0) - r0) ** 2)
                for k in ks if k <= n))
        eqm.append(np.mean(acc))

    eqm = np.array(eqm)
    resultados[d] = eqm
    expoente = np.polyfit(np.log(ns), np.log(eqm), 1)[0]      # (b) e (c)
    print(f"d={d:2d}: " + "  ".join(f"{v:.5f}" for v in eqm)
          + f"   expoente {expoente:+.3f}   cota {-2 / (2 + d):+.3f}")

Deve imprimir:

```
d= 1: 0.00761  0.00470  0.00272  0.00134  0.00111   expoente -0.738   cota -0.667
d= 5: 0.13169  0.10313  0.08026  0.05996  0.04535   expoente -0.386   cota -0.286
d=10: 0.23894  0.20745  0.17961  0.15840  0.13452   expoente -0.205   cota -0.167
```

**O que reproduz.** O expoente cai em módulo conforme $p$ cresce, exatamente como
o teorema prevê: $-0{,}74$, $-0{,}39$, $-0{,}21$. E a razão entre os extremos
bate bem: medimos $0{,}738/0{,}205 = 3{,}6$, contra os $0{,}667/0{,}167 = 4{,}0$
previstos. A degradação também aparece no nível do erro — com $n=4000$, o erro em
$p=10$ é **121 vezes** o erro em $p=1$.

**O que não reproduz.** Nenhum dos três expoentes é igual à cota, e em $p=5$ e
$p=10$ o erro medido cai **mais rápido** do que ela.

E isso está certo. O teorema dá uma **cota superior**, válida para toda função
$L$-Lipschitz — inclusive as mais difíceis, que oscilam o máximo permitido em
cada vizinhança. A nossa $r$ é uma soma de senos: infinitamente diferenciável e
bem mais fácil que o pior caso. Métodos de vizinhança aproveitam suavidade extra,
e a taxa real fica melhor que a garantida.

A moral vale para o curso inteiro: uma cota superior que não é atingida **não
está errada**. Se a medição desse um expoente *pior* que a cota, aí sim haveria
algo a explicar — ou no experimento, ou no teorema.

---
## Exercício 4 — a fuga: supor estrutura

A maldição vale para quem não supõe nada sobre $r$. A nossa $r$ é, por
construção, **aditiva**: uma soma de funções de uma variável cada.

Um método que sabe disso pode estimar cada parcela separadamente — e cada uma é
um problema unidimensional, imune à maldição. É o que faz um modelo aditivo com
*splines*: `SplineTransformer` gera as bases de cada coordenada isoladamente, e a
regressão linear combina todas.

Compare os dois em $p=10$.

In [ ]:
d, n = 10, 2000
rng = np.random.default_rng(2026)
erros_knn, erros_aditivo = [], []

for _ in range(6):
    X = rng.uniform(0, 1, size=(n, d))
    y = r(X) + rng.normal(0, SIGMA, size=n)
    X0 = rng.uniform(0, 1, size=(3000, d))
    r0 = r(X0)

    erros_knn.append(min(
        np.mean((KNeighborsRegressor(n_neighbors=int(k)).fit(X, y).predict(X0) - r0) ** 2)
        for k in ks if k <= n))

    aditivo = make_pipeline(
        SplineTransformer(n_knots=8, degree=3),                # (a) nos por coordenada
        LinearRegression(),
    ).fit(X, y)
    erros_aditivo.append(np.mean((aditivo.predict(X0) - r0) ** 2))   # (b)

print(f"KNN pleno (melhor k):        EQM {np.mean(erros_knn):.5f}")
print(f"aditivo (splines por coord): EQM {np.mean(erros_aditivo):.5f}")
print(f"razao: {np.mean(erros_knn) / np.mean(erros_aditivo):.1f}x")   # (c)

Deve imprimir:

```
KNN pleno (melhor k):        EQM 0.15765
aditivo (splines por coord): EQM 0.00467
razao: 33.8x
```

O modelo aditivo erra **34 vezes menos**, com os mesmos 2 000 pontos. E note o
tamanho do efeito: o erro do aditivo em $p=10$ com $n=2000$ (0,0047) é da ordem
do erro do KNN em $p=1$ com $n=250$ (0,0076) — ou seja, a suposição de
aditividade devolveu ao problema de dimensão 10 o comportamento de um problema
unidimensional.

O que comprou esse ganho não foi mais dado nem um algoritmo mais esperto: foi uma
**suposição**. E ela é verdadeira aqui porque nós escrevemos $r$ como uma soma —
num problema real, seria uma aposta.

É a lição central da aula, e vale para todo o resto do curso: a maldição da
dimensionalidade é o preço de não supor nada. Regressão linear supõe muito
(Aula 02), o Lasso supõe esparsidade, o modelo aditivo supõe ausência de
interações, e as árvores da Aula 06 supõem que $r$ é aproximadamente constante
por blocos. Cada uma dessas suposições é uma troca de viés por taxa.

> **Sua vez.** Troque a função de regressão por
> $r(x) = \operatorname{sen}(2\pi x_1 x_2)$, que **não** é aditiva, e repita a
> comparação em $p=10$. O modelo aditivo continua ganhando?

---
## O que ficou

| Exercício | O que você mediu |
|---|---|
| 1 | com uma casca de 5%, em $p=50$ **99,5%** dos pontos estão nela — e a fórmula acerta em todas as dimensões |
| 2 | o volume da bola unitária cresce até $p=5$ e depois desaba: $10^{-13}$ em $p=50$ |
| 2 | de $p=20$ em diante, as duas fórmulas subestimam o raio — porque a bola já não cabe no cubo |
| 3 | o expoente medido cai como previsto ($-0{,}74$, $-0{,}39$, $-0{,}21$), mas é sempre **melhor** que a cota |
| 4 | supor aditividade divide o erro por **34** em $p=10$, sem um dado a mais |

**A seguir.** A Aula 06 traz uma família que faz uma suposição diferente — $r$
aproximadamente constante por blocos — e um truque para reduzir a variância de
estimadores instáveis sem tocar no viés deles.